# Credit Events and Restructuring

**Purpose:** Show how the `finstack.valuations` restructuring utilities model three common distressed-credit workflows.

**Prerequisites:** Basic familiarity with capital-structure priority, exchange offers, and liability management exercises.

**In this notebook:** We run a recovery waterfall, compare hold-versus-tender economics for an exchange offer, and quantify an open-market repurchase LME.


## Concept

These helpers are useful when you want a fast first-pass answer to questions like:

1. Who recovers what under a stressed distributable value?
2. Is an exchange offer economically attractive to holders?
3. How much debt reduction does a discounted buyback create?


In [ ]:
from finstack.valuations import (
    analyze_exchange_offer,
    analyze_lme,
    execute_recovery_waterfall,
)


def banner(title: str) -> None:
    print(f"\n{'=' * 72}")
    print(title)
    print('=' * 72)


def fmt_money(value: float) -> str:
    return f"${value:,.0f}"


def fmt_pct(value: float) -> str:
    return f"{value * 100:.2f}%"


## Waterfall, exchange offer, and LME

The next cell adapts the flat script into three sections so you can inspect each result separately. This is a good pricing-adjacent notebook because the outputs are decision-support numbers rather than full marked-to-market valuations.


In [ ]:
banner("Recovery waterfall")
claims = [
    {
        "id": "first_lien_tl",
        "label": "First Lien Term Loan",
        "seniority": "first_lien",
        "principal": 50_000_000.0,
        "accrued": 1_000_000.0,
        "collateral_value": 40_000_000.0,
        "haircut": 0.0,
    },
    {
        "id": "sr_unsec_notes",
        "label": "Senior Unsecured Notes",
        "seniority": "senior_unsecured",
        "principal": 80_000_000.0,
        "accrued": 2_000_000.0,
    },
    {
        "id": "sub_notes",
        "label": "Subordinated Notes",
        "seniority": "subordinated",
        "principal": 40_000_000.0,
        "accrued": 1_000_000.0,
    },
    {
        "id": "common_equity",
        "label": "Common Equity",
        "seniority": "equity",
        "principal": 0.01,
    },
]

waterfall = execute_recovery_waterfall(
    total_value=100_000_000.0,
    currency="USD",
    claims=claims,
    allocation_mode="pro_rata",
)
print(f"Total distributed: {fmt_money(waterfall['total_distributed'])}")
print(f"Residual         : {fmt_money(waterfall['residual'])}")
for row in waterfall["per_claim_recovery"]:
    print(
        f"  {row['id']:<20} claim={fmt_money(row['total_claim'])} "
        f"recovery={fmt_money(row['total_recovery'])} rate={fmt_pct(row['recovery_rate'])}"
    )

banner("Exchange offer")
exchange = analyze_exchange_offer(
    old_pv=45.0,
    new_pv=80.0,
    consent_fee=2.0,
    equity_sweetener_value=0.0,
    exchange_type="discount",
)
print(f"Tender total     : ${exchange['tender_total']:.2f}")
print(f"Delta NPV        : ${exchange['delta_npv']:+.2f}")
print(f"Breakeven recov. : {fmt_pct(exchange['breakeven_recovery'])}")
print(f"Tender?          : {exchange['tender_recommended']}")

banner("Open-market repurchase LME")
lme = analyze_lme(
    lme_type="open_market",
    notional=200_000_000.0,
    repurchase_price_pct=0.60,
    opt_acceptance_pct=0.40,
    ebitda=25_000_000.0,
)
print(f"Cash cost        : {fmt_money(lme['cost'])}")
print(f"Notional retired : {fmt_money(lme['notional_reduction'])}")
print(f"Discount capture : {fmt_money(lme['discount_capture'])}")
print(f"Holder impact    : {fmt_pct(lme['remaining_holder_impact_pct'])}")
if lme['leverage_impact'] is not None:
    leverage = lme['leverage_impact']
    print(f"Pre leverage     : {leverage['pre_leverage']:.2f}x")
    print(f"Post leverage    : {leverage['post_leverage']:.2f}x")


## Takeaways

- `execute_recovery_waterfall()` is the quickest way to turn a stressed enterprise value into per-claim recoveries.
- `analyze_exchange_offer()` reframes a restructuring proposal in holder economics rather than issuer rhetoric.
- `analyze_lme()` is useful for measuring debt reduction and leverage change from discounted liability-management moves.


In [ ]:
{
    "waterfall_apr_satisfied": waterfall["apr_satisfied"],
    "exchange_delta_npv": round(exchange["delta_npv"], 2),
    "lme_discount_capture": round(lme["discount_capture"], 2),
}
